# Setup

## Load packages

In [1]:
# Load up necessary packages. 
import os
import glob
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Check GPU availability. 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Import my custom utils.
import utils
importlib.reload(utils)

Using device: cuda
GPU: NVIDIA GeForce RTX 3090


<module 'utils' from '/tscc/projects/ps-yeolab3/kflanagan/plip_plop/machine_learning_prototype/utils.py'>

## Load results

In [2]:
INPUT_DIR = "/tscc/lustre/ddn/scratch/kflanagan/plip_plop_results/processed_inputs/"

COMPARISON_TABLE = (
    "/tscc/nfs/home/kflanagan/projects/plip_plop/"
    "machine_learning_prototype/encode_initial_30_comparisons.tsv"
)

comparison_table = pd.read_csv(COMPARISON_TABLE, sep="\t")

# Running pytorch

## Data setup

### Create metadata

In [3]:
all_signals = []
all_metadata = []

for comparison_id, row in comparison_table.iterrows():
    experiment_A = row["experiment_A"]
    experiment_B = row["experiment_B"]

    npz_file = os.path.join(INPUT_DIR, f"{experiment_A}_{experiment_B}.npz")

    if not os.path.exists(npz_file):
        print(f"Missing: {npz_file}")
        continue

    data = np.load(npz_file)

    # Load the final abundance-corrected, smoothed signal.
    signals = data["signals"].astype(np.float32)
    n_windows = len(signals)

    all_signals.append(signals)

    metadata = pd.DataFrame({
        "comparison_id": comparison_id,
        "category": row["category"],
        "experiment_A": experiment_A,
        "experiment_B": experiment_B,
        "chrom": data["chrom"],
        "start": data["start"],
        "end": data["end"],
        "strand": data["strand"],
        "region_id": data["region_id"],
        "block_number": data["block_number"],
        "signal_A": data["total_signal_A"],
        "signal_B": data["total_signal_B"],
        "total_signal": data["total_signal"]
    })

    all_metadata.append(metadata)

In [4]:
signals = np.concatenate(all_signals, axis=0)
metadata = pd.concat(all_metadata, ignore_index=True)

### Weighting

In [5]:
comparison_counts = (
    metadata
    .groupby(["comparison_id", "category", "experiment_A", "experiment_B"])
    .size()
    .reset_index(name="n_windows")
    .sort_values("n_windows", ascending=False)
)

In [6]:
# Set weighting alpha to 0.5 (square root).
alpha = 0.5

# Create the comparison weights.
comparison_counts["comparison_weight"] = comparison_counts["n_windows"] ** (alpha - 1)

# Create a dictionary for quickly grabbing each weight for each comparison.
weight_map = dict(zip(comparison_counts["comparison_id"], comparison_counts["comparison_weight"]))

# Add comparison weights to metadata.
metadata["comparison_weight"] = metadata["comparison_id"].map(weight_map)

# Use log-transformed total IP signal for window weighting.
metadata["signal_weight"] = np.log1p(metadata["total_signal"])

# Normalize signal weights within each comparison.
metadata["signal_weight"] = (
    metadata["signal_weight"] /
    metadata.groupby("comparison_id")["signal_weight"].transform("mean")
)

# Create the final weight.
metadata["weight"] = metadata["comparison_weight"] * metadata["signal_weight"]

## Train/val split setup. 

In [7]:
# Create a combined identifier from comparison and region. 
metadata["region_key"] = (
    metadata["comparison_id"].astype(str)
    + "_"
    + metadata["region_id"].astype(str)
)

In [8]:
# Setup random seed. 
rng = np.random.default_rng(42)

# Initialize masking vector. 
train_mask = np.zeros(len(metadata), dtype=bool)
val_mask = np.zeros(len(metadata), dtype=bool)

# Build mask. 
for comparison_id in metadata["comparison_id"].unique():
    comparison_rows = metadata["comparison_id"] == comparison_id

    comparison_regions = (
        metadata.loc[comparison_rows, "region_key"]
        .unique()
        .copy()
    )

    rng.shuffle(comparison_regions)

    split = int(len(comparison_regions) * 0.8)

    train_regions = comparison_regions[:split]
    val_regions = comparison_regions[split:]

    train_mask |= metadata["region_key"].isin(train_regions)
    val_mask |= metadata["region_key"].isin(val_regions)

/scratch/kflanagan/job_12282982/ipykernel_3941768/2334664090.py:18: UserWarning: you are shuffling a 'StringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  rng.shuffle(comparison_regions)


In [9]:
# Apply mask to the signals.  
train_signals = signals[train_mask]
val_signals = signals[val_mask]

# Apply mask to the metadata. 
train_metadata = metadata.loc[train_mask].reset_index(drop=True)
val_metadata = metadata.loc[val_mask].reset_index(drop=True)

print("Training windows:", len(train_signals))
print("Validation windows:", len(val_signals))

Training windows: 574631
Validation windows: 143119


## Setup data loaders. 

In [10]:
# Create special data class for machine learning. 
class RBPWindowDataset(Dataset):
    def __init__(self, signals, weights):
        self.signals = torch.tensor(signals, dtype=torch.float32)
        self.weights = torch.tensor(weights, dtype=torch.float32)

    def __len__(self):
        return(len(self.signals))

    def __getitem__(self, idx):
        return(self.signals[idx], self.weights[idx])

train_weights = train_metadata["weight"].to_numpy(dtype=np.float32)
val_weights = val_metadata["weight"].to_numpy(dtype=np.float32)

train_dataset = RBPWindowDataset(train_signals, train_weights)
val_dataset = RBPWindowDataset(val_signals, val_weights)

train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1024, shuffle=False)

In [11]:
batch_signals, batch_weights = next(iter(train_loader))

print("Signals:", batch_signals.shape)
print("Weights:", batch_weights.shape)
print("First few weights:", batch_weights[:10])

Signals: torch.Size([1024, 2, 300])
Weights: torch.Size([1024])
First few weights: tensor([0.0053, 0.0098, 0.0109, 0.0098, 0.0086, 0.0065, 0.0034, 0.0040, 0.0037,
        0.0030])


## Model setup

In [12]:
# Set model as autoencode
model = utils.SimpleAutoencoder(latent_dim=64).to(device)

# Adam optimizer (what is adam?)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## Run model

In [13]:
train_losses = []
val_losses = []

best_val_loss = float("inf")
best_model_state = None

num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0
    total_train_windows = 0

    for batch_signals, batch_weights in train_loader:
        batch_signals = batch_signals.to(device)
        batch_weights = batch_weights.to(device)

        optimizer.zero_grad()

        reconstruction = model(batch_signals)
        loss = utils.weighted_mse_loss(reconstruction, batch_signals, batch_weights)

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item() * len(batch_signals)
        total_train_windows += len(batch_signals)

    average_train_loss = total_train_loss / total_train_windows

    model.eval()
    total_val_loss = 0
    total_val_windows = 0

    with torch.no_grad():
        for batch_signals, batch_weights in val_loader:
            batch_signals = batch_signals.to(device)
            batch_weights = batch_weights.to(device)

            reconstruction = model(batch_signals)
            loss = utils.weighted_mse_loss(reconstruction, batch_signals, batch_weights)

            total_val_loss += loss.item() * len(batch_signals)
            total_val_windows += len(batch_signals)

    average_val_loss = total_val_loss / total_val_windows

    train_losses.append(average_train_loss)
    val_losses.append(average_val_loss)

    if average_val_loss < best_val_loss:
        best_val_loss = average_val_loss
        best_model_state = {
            key: value.cpu().clone()
            for key, value in model.state_dict().items()
        }

    print(
        f"Epoch {epoch + 1}: "
        f"train = {average_train_loss:.6f}, "
        f"validation = {average_val_loss:.6f}"
    )

Epoch 1: train = 0.037200, validation = 0.004929
Epoch 2: train = 0.003739, validation = 0.002849
Epoch 3: train = 0.002423, validation = 0.002139
Epoch 4: train = 0.001977, validation = 0.001832
Epoch 5: train = 0.001813, validation = 0.001643
Epoch 6: train = 0.001721, validation = 0.001544
Epoch 7: train = 0.001663, validation = 0.001539
Epoch 8: train = 0.001629, validation = 0.001523
Epoch 9: train = 0.001593, validation = 0.001505
Epoch 10: train = 0.001555, validation = 0.001542
Epoch 11: train = 0.001538, validation = 0.001536
Epoch 12: train = 0.001514, validation = 0.001389
Epoch 13: train = 0.001509, validation = 0.001579
Epoch 14: train = 0.001481, validation = 0.001610
Epoch 15: train = 0.001471, validation = 0.001466
Epoch 16: train = 0.001463, validation = 0.001418
Epoch 17: train = 0.001443, validation = 0.001413
Epoch 18: train = 0.001430, validation = 0.001412
Epoch 19: train = 0.001419, validation = 0.001400
Epoch 20: train = 0.001422, validation = 0.001342


In [14]:
# Loads best model, not just last model. 
model.load_state_dict(best_model_state)
model = model.to(device)

# Save model for later use. 
torch.save(model.state_dict(), "/tscc/nfs/home/kflanagan/scratch/plip_plop_results/many_IN_RBP_baseline.pt")